# Pelatihan Model Random Forest

### Deskripsi Tugas
Menggunakan **data training hasil TF-IDF dari Najmi** (file `.npz` dan `.csv`) untuk melatih model **Random Forest**.

Algoritma ini membangun banyak *decision tree* secara acak lalu menggabungkan hasilnya:
> *"Dari ratusan pohon keputusan yang masing-masing memilih fitur kata secara acak — mana kelas yang paling banyak dipilih?"*

**Input (dari Najmi):**
- `X_train.npz` — matriks TF-IDF data training (7683 × 5000)
- `X_test.npz`  — matriks TF-IDF data testing  (1921 × 5000)
- `y_train.csv` — label training
- `y_test.csv`  — label testing

**Output:**
- Model Random Forest terbaik tersimpan di `model_random_forest.pkl`
- Hasil eksperimen hyperparameter di `hasil_eksperimen_random_forest.xlsx`


## 1. Import Library

In [ ]:
import pickle
import scipy.sparse as sp
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

print("Semua library berhasil diimport!")

Semua library berhasil diimport!


## 2. Load Data dari Najmi
Load file hasil TF-IDF yang sudah disiapkan oleh Najmi.
Pastikan keempat file berada di folder yang sama dengan notebook ini.


In [ ]:
# Load X_train
train_data = np.load("X_train.npz/data.npy")
train_indices = np.load("X_train.npz/indices.npy")
train_indptr = np.load("X_train.npz/indptr.npy")
train_shape = tuple(np.load("X_train.npz/shape.npy"))

X_train = sp.csr_matrix((train_data, train_indices, train_indptr), shape=train_shape)

# Load X_test
test_data = np.load("X_test.npz/data.npy")
test_indices = np.load("X_test.npz/indices.npy")
test_indptr = np.load("X_test.npz/indptr.npy")
test_shape = tuple(np.load("X_test.npz/shape.npy"))

X_test = sp.csr_matrix((test_data, test_indices, test_indptr), shape=test_shape)

# Load label
y_train_df = pd.read_csv("y_train.csv")
y_test_df = pd.read_csv("y_test.csv")

# Print
print("=== DATA BERHASIL DILOAD ===")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train_df.shape}")
print(f"y_test  : {y_test_df.shape}")

=== DATA BERHASIL DILOAD ===
X_train : (7683, 5000)
X_test  : (1921, 5000)
y_train : (7683, 1)
y_test  : (1921, 1)


## 3. Bersihkan & Standardisasi Label
Sama seperti Muthia, ada variasi penulisan label, perlu diseragamkan dulu.

In [ ]:
def bersihkan_label(series):
    """Standardisasi label: strip spasi, uppercase, lalu encode Y=1 / N=0."""
    bersih = series.astype(str).str.strip().str.upper()
    return (bersih == 'Y').astype(int)

y_train = bersihkan_label(y_train_df.iloc[:, 0])
y_test  = bersihkan_label(y_test_df.iloc[:, 0])

print("Setelah standardisasi:")
print(f"y_train — Beli (1): {y_train.sum():,}  |  Tidak Beli (0): {(y_train==0).sum():,}")
print(f"y_test  — Beli (1): {y_test.sum():,}   |  Tidak Beli (0): {(y_test==0).sum():,}")
print()
print(f"Proporsi kelas Beli di train : {y_train.mean()*100:.1f}%")
print(f"Proporsi kelas Beli di test  : {y_test.mean()*100:.1f}%")

Setelah standardisasi:
y_train — Beli (1): 6,814  |  Tidak Beli (0): 869
y_test  — Beli (1): 1,715   |  Tidak Beli (0): 206

Proporsi kelas Beli di train : 88.7%
Proporsi kelas Beli di test  : 89.3%


## 4. Teori Random Forest

### Konsep Dasar
Random Forest adalah algoritma **ensemble learning** berbasis banyak *decision tree*.
$$\hat{y} = \text{mode}\left(\hat{y}_1,\ \hat{y}_2,\ \ldots,\ \hat{y}_T\right)$$

Di mana $\hat{y}_t$ adalah prediksi pohon ke-$t$, dan kelas akhir dipilih berdasarkan **voting mayoritas**.

### Cara Kerja
1. **Bootstrap Sampling** — setiap pohon dilatih pada subset acak (dengan pengembalian) dari data training
2. **Random Feature Selection** — setiap node hanya mempertimbangkan subset acak dari fitur (kata TF-IDF)
3. **Voting** — hasil prediksi semua pohon digabungkan dengan voting mayoritas

### Keunggulan vs Naïve Bayes
| Aspek | Naïve Bayes | Random Forest |
|-------|------------|---------------|
| Asumsi fitur | Independen | Tidak ada |
| Kecepatan training | Sangat cepat | Lebih lambat |
| Kemampuan non-linear | Terbatas | Kuat |
| Interpretabilitas | Tinggi | Sedang |
| Imbalanced data | Perlu penanganan | Lebih robust |

### Parameter yang Diuji

| Parameter | Deskripsi | Nilai yang Diuji |
|-----------|-----------|-----------------|
| `n_estimators` | Jumlah pohon di forest | 100, 200 |
| `max_depth` | Kedalaman maksimal tiap pohon | None, 10, 20 |
| `class_weight` | Bobot kelas (menangani imbalance) | 'balanced' |

`max_depth=None` artinya pohon tumbuh bebas sampai semua node murni.

## 5. Eksperimen Hyperparameter Random Forest
Menguji: 2 n_estimators × 3 max_depth = **6 model**


In [ ]:

# Konfigurasi eksperimen
configs = [
    {"n_estimators": 100, "max_depth": None},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 100, "max_depth": 20},
    {"n_estimators": 200, "max_depth": None},
    {"n_estimators": 200, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 20},
]

hasil = []

print(f"{'Model':<40} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("-" * 90)

for cfg in configs:
    n_est   = cfg["n_estimators"]
    m_depth = cfg["max_depth"]
    depth_str = str(m_depth) if m_depth is not None else "None"
    nama = f"RF (n={n_est}, depth={depth_str})"

    model = RandomForestClassifier(
        n_estimators=n_est,
        max_depth=m_depth,
        class_weight='balanced',   # menangani imbalanced data
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_proba)

    hasil.append({
        "Model": nama, "n_estimators": n_est,
        "max_depth": depth_str,
        "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1": f1, "AUC": auc
    })
    print(f"{nama:<40} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {auc:>10.4f}")

hasil_df = pd.DataFrame(hasil).sort_values("F1", ascending=False).reset_index(drop=True)
hasil_df.index += 1


Model                                      Accuracy  Precision     Recall         F1        AUC
------------------------------------------------------------------------------------------
RF (n=100, depth=None)                       0.9297     0.9355     0.9895     0.9617     0.9356
RF (n=100, depth=10)                         0.8673     0.9704     0.8781     0.9219     0.9014
RF (n=100, depth=20)                         0.8917     0.9614     0.9155     0.9379     0.9133
RF (n=200, depth=None)                       0.9308     0.9346     0.9918     0.9624     0.9375
RF (n=200, depth=10)                         0.8693     0.9710     0.8799     0.9232     0.9036
RF (n=200, depth=20)                         0.8995     0.9623     0.9236     0.9426     0.9109


## 6. Tabel Perbandingan Semua Model


In [ ]:
display_df = hasil_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']:
    display_df[col] = display_df[col].map(lambda x: f"{x:.4f}")

print("PERINGKAT MODEL (urut F1-Score tertinggi)")
print(display_df.to_string())

PERINGKAT MODEL (urut F1-Score tertinggi)
                    Model  n_estimators max_depth Accuracy Precision  Recall      F1     AUC
1  RF (n=200, depth=None)           200      None   0.9308    0.9346  0.9918  0.9624  0.9375
2  RF (n=100, depth=None)           100      None   0.9297    0.9355  0.9895  0.9617  0.9356
3    RF (n=200, depth=20)           200        20   0.8995    0.9623  0.9236  0.9426  0.9109
4    RF (n=100, depth=20)           100        20   0.8917    0.9614  0.9155  0.9379  0.9133
5    RF (n=200, depth=10)           200        10   0.8693    0.9710  0.8799  0.9232  0.9036
6    RF (n=100, depth=10)           100        10   0.8673    0.9704  0.8781  0.9219  0.9014


## 7. Pilih Model Terbaik


In [ ]:

best_row  = hasil_df.iloc[0]
best_name = best_row["Model"]

print("=" * 55)
print("MODEL TERPILIH")
print("=" * 55)
print(f"  Nama         : {best_name}")
print(f"  n_estimators : {int(best_row['n_estimators'])}")
print(f"  max_depth    : {best_row['max_depth']}")
print(f"  F1-Score     : {best_row['F1']:.4f}")
print(f"  AUC-ROC      : {best_row['AUC']:.4f}")
print(f"  Accuracy     : {best_row['Accuracy']:.4f}")
print(f"  Precision    : {best_row['Precision']:.4f}")
print(f"  Recall       : {best_row['Recall']:.4f}")

# Instansiasi ulang model terbaik
best_n_est   = int(best_row['n_estimators'])
best_depth   = None if best_row['max_depth'] == 'None' else int(best_row['max_depth'])

best_model = RandomForestClassifier(
    n_estimators=best_n_est,
    max_depth=best_depth,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
best_model.fit(X_train, y_train)
print()
print(f"Model dilatih pada {X_train.shape[0]:,} data training dari Najmi.")


MODEL TERPILIH
  Nama         : RF (n=200, depth=None)
  n_estimators : 200
  max_depth    : None
  F1-Score     : 0.9624
  AUC-ROC      : 0.9375
  Accuracy     : 0.9308
  Precision    : 0.9346
  Recall       : 0.9918

Model dilatih pada 7,683 data training dari Najmi.


## 8. Classification Report Lengkap

In [ ]:
y_pred_best = best_model.predict(X_test)

print("CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(y_test, y_pred_best,
                             target_names=["Tidak Beli (N)", "Beli (Y)"]))

cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:")
print(f"                    Pred: Tidak Beli   Pred: Beli")
print(f"  Actual: Tidak Beli       {tn:>5}          {fp:>5}")
print(f"  Actual: Beli             {fn:>5}          {tp:>5}")
print()
print(f"  True Negative  (TN) = {tn:,}")
print(f"  False Positive (FP) = {fp:,}")
print(f"  False Negative (FN) = {fn:,}")
print(f"  True Positive  (TP) = {tp:,}")

CLASSIFICATION REPORT
                precision    recall  f1-score   support

Tidak Beli (N)       0.86      0.42      0.57       206
      Beli (Y)       0.93      0.99      0.96      1715

      accuracy                           0.93      1921
     macro avg       0.90      0.71      0.76      1921
  weighted avg       0.93      0.93      0.92      1921

Confusion Matrix:
                    Pred: Tidak Beli   Pred: Beli
  Actual: Tidak Beli          87            119
  Actual: Beli                14           1701

  True Negative  (TN) = 87
  False Positive (FP) = 119
  False Negative (FN) = 14
  True Positive  (TP) = 1,701


## 9. Cross-Validation 5-Fold


In [ ]:
cv_scores = cross_val_score(best_model, X_train, y_train,
                             cv=5, scoring='f1', n_jobs=-1)

print("CROSS-VALIDATION (5-Fold) — F1-Score")
print("=" * 45)
for i, score in enumerate(cv_scores, 1):
    bar = '█' * int(score * 30)
    print(f"  Fold {i}: {score:.4f}  {bar}")
print()
print(f"  Rata-rata : {cv_scores.mean():.4f}")
print(f"  Std Dev   : {cv_scores.std():.4f}")
if cv_scores.std() < 0.02:
    print("Model STABIL — std dev rendah, tidak overfitting")
else:
    print("Pertimbangkan tuning lebih lanjut")

CROSS-VALIDATION (5-Fold) — F1-Score
  Fold 1: 0.9541  ████████████████████████████
  Fold 2: 0.9579  ████████████████████████████
  Fold 3: 0.9486  ████████████████████████████
  Fold 4: 0.9530  ████████████████████████████
  Fold 5: 0.9557  ████████████████████████████

  Rata-rata : 0.9539
  Std Dev   : 0.0031
Model STABIL — std dev rendah, tidak overfitting


## 10. Feature Importance (Top 20 Kata Paling Berpengaruh)
Random Forest bisa menunjukkan kata TF-IDF mana yang paling penting
untuk prediksi — ini bonus yang tidak ada di Naïve Bayes!
# Catatan:

fitur_names harus di-load dari vectorizer Najmi

Jika ada file vectorizer, uncomment baris di bawah:
# import pickle
vectorizer = pickle.load(open("tfidf_vectorizer.pkl", "rb"))

fitur_names = vectorizer.get_feature_names_out()

importances = best_model.feature_importances_

top_idx = importances.argsort()[::-1][:20]

print("Top 20 Kata Paling Berpengaruh:")

for rank, idx in enumerate(top_idx, 1):

print(f"  {rank:>2}. {fitur_names[idx]:<20} importance: {importances[idx]:.6f}")


In [ ]:
print("Feature importance tersedia di best_model.feature_importances_")
print(f"Jumlah fitur TF-IDF: {X_train.shape[1]:,}")
print("(Uncomment blok di atas jika file vectorizer Najmi tersedia)")


Feature importance tersedia di best_model.feature_importances_
Jumlah fitur TF-IDF: 5,000
(Uncomment blok di atas jika file vectorizer Najmi tersedia)


## 11. Simpan Model ke File `.pkl`
Model disimpan agar bisa dipakai untuk selanjutnya


In [ ]:
with open("model_random_forest.pkl", "wb") as f:
    pickle.dump(best_model, f)
print("✅ model_random_forest.pkl          — tersimpan")

hasil_df.to_excel("hasil_eksperimen_random_forest.xlsx", index=False)
print("✅ hasil_eksperimen_random_forest.xlsx — tersimpan")

print()
print("Cara load model nanti:")
print("  import pickle")
print("  model = pickle.load(open('model_random_forest.pkl', 'rb'))")
print("  y_pred = model.predict(X_test_baru)")


✅ model_random_forest.pkl          — tersimpan
✅ hasil_eksperimen_random_forest.xlsx — tersimpan

Cara load model nanti:
  import pickle
  model = pickle.load(open('model_random_forest.pkl', 'rb'))
  y_pred = model.predict(X_test_baru)
